In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated/checkpoints/post_cell_21.pickle

trying: ['i_1', 'i_3']
me:  11
me:  13
trying: ['i_1', 'i_3']
me:  11
me:  13
trying: ['test_txt']
me:  1
trying: ['train_txt']
me:  1
trying: ['pd']
me:  0
trying: ['glob']
me:  0
trying: ['np']
me:  0
trying: ['text']
me:  24
trying: ['sample_discourse_id']
me:  2
trying: ['myword']
me:  12
trying: ['dt']
me:  24
trying: ['len_dict']
me:  12
trying: ['word_dict']
me:  12
trying: ['mylen']
me:  12
trying: ['cols_to_display']
me:  14
trying: ['myid']
me:  12
trying: ['df1']
me:  24
trying: ['style']
me:  0
trying: ['data']
me:  12
trying: ['CountVectorizer']
me:  0
trying: ['txt_file']
me:  12
trying: ['cols_to_merge']
me:  13
trying: ['ax1']
me:  8
trying: ['plt']
me:  0
trying: ['t']
me:  12
trying: ['train']
me:  24
trying: ['FuncFormatter']
me:  0
trying: ['sample_submission']
me:  2
trying: ['ax2']
me:  8
trying: ['warnings']
me:  0
trying: ['IREWR_tmp2']
me:  24
trying: ['total_gaps']
me:  24
trying: ['last_ones']
me:  24
trying: ['train_first_last']
me:  11
trying: ['stopwords']

In [4]:
%%RecordEvent
%%time
### cell 22 ###
def get_n_grams(n_grams, top_n=10):
    # Vectorize all texts in one go rather than per discourse type
    texts = train['discourse_text'].tolist()
    types = train['discourse_type'].values
    vectorizer = CountVectorizer(
        lowercase=True,
        stop_words='english',
        ngram_range=(n_grams, n_grams)
    )
    bag = vectorizer.fit_transform(texts)
    # Build a DataFrame of shape (n_docs, n_ngrams)
    feature_names = vectorizer.get_feature_names_out()
    bow_df = pd.DataFrame(
        bag.toarray(),
        columns=feature_names
    )
    bow_df['discourse_type'] = types
    # Sum counts per discourse_type (this will be a single GPU groupby)
    summed = bow_df.groupby('discourse_type').sum()
    # For each discourse_type, pick the top_n n-grams
    records = []
    for d_type, row in summed.iterrows():
        top_series = row.nlargest(top_n)
        for word, count in top_series.items():
            records.append((d_type, word, int(count)))
    return pd.DataFrame(records, columns=['Discourse_type', 'words', 'counts'])

# compute bigrams
gram_counts = get_n_grams(n_grams=2, top_n=10)
gram_counts.head()

CPU times: user 1.95 s, sys: 30.2 ms, total: 1.98 s
Wall time: 1.98 s


,Discourse_type,words,counts
0,Claim,cell phone,5
1,Claim,cell phones,3
2,Claim,phone driving,3
3,Claim,car accidents,2
4,Claim,driving phone,2


In [5]:
%Checkpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/cell_19/checkpoints/post_cell_22_try_4.pickle

migration speed (bps): 771624551.1968536
---------------------------
variables to migrate:
orig_output 16
counts_dict 696
other_words_to_take_out 152
add_gap_rows 144
IREWR_tmp 48
stop_english 1688
train 48
glob 144
total_gaps 48
IREWR_tmp2 48
tqdm 1072
test_txt 120
np 72
train_txt 126104
print_colored_essay 144
add_gap_rows_2 144
IREWR_plug_1 61
sample_discourse_id 32
IREWR_plug_2 61
txt_file 208
myword 28
len_dict 589920
word_dict 589920
mylen 28
style 72
ax1 48
myid 61
CountVectorizer 1480
i_3 28
cols_to_merge 120
data 2813
ax2 48
sample_submission 48
cols_to_display 120
FuncFormatter 1072
plt 72
get_n_grams 144
pd 72
warnings 72
train_first_last 48
i_1 28
counter 28
ax 48
t 166
train_first 48
last_ones 48
train_last 48
text 408
dt 57
gram_counts 48
stopwords 48
all_gaps 48
av_per_essay 48
df1 48
df 48
keys 120
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/notebooks/erikbruin/nlp-o

In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['train_txt', 'train', 'sample_submission', 'stopwords']
Intermediate variables ['test_txt']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['train', 'sample_discourse_id']
Intermediate variables ['sample_submission']
Future variables ['train_txt']
Modified dataframes
  train
    Input columns: set()
    Changed columns: {'discourse_type_num', 'predictionstring', 'discourse_text', 'id', 'discourse_start', 'discourse_id', 'discourse_end', 'discourse_type'}
    Created columns: set()
    Deleted columns: set()
  sample_submission
    Input columns: set()
    Changed columns: {'predictionstring', 'class', 'id'}
    Created columns: set()
    Deleted columns: set()
======= Cell 2 =======
Input variables []
Active variables ['cols_to_display', 'train']
Intermediate variables []
Future variables ['train_txt', 'sample_discourse_id']
Modified dataframes
  train
    Input columns: set(

In [7]:

with open("/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/opt_cell_exec_info_22_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[22], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
assert False, 'bigrams should be rewritten in the optimized code.'